# Security Incident: Data Cleaning & Initial EDA

**Dataset:** Microsoft Security Incident Prediction (`GUIDE_Train.csv`)
**Goal of this notebook:** load the raw data, understand its structure, resolve data-quality
questions (unclear zeros, identifier columns, duplicates), and produce a clean CSV that
downstream notebooks (categorical EDA, continuous EDA, modeling) can load without
re-deriving these decisions.

**Topics in this project:**
1. `01_data_cleaning` *(this notebook)*
2. `02_categorical_eda`
3. `03_continuous_eda`
4. `04_modeling`

> **Note on columns investigated below:** several columns turned out to use a dominant
> "sentinel" value standing in for "not applicable to this entity type" rather than a real
> measurement. These are confirmed in Section 8 and converted to `NaN`; ambiguous cases are
> left unmodified and can be tested with/without during modeling.


## 0. Dataset Background (from source documentation)

A few facts from Microsoft's GUIDE dataset documentation that inform decisions later in
this notebook:

- **Anonymization:** values are pseudo-anonymized with SHA-1 hashing, then the hashed
  values are replaced with randomly generated identifiers. Noise is also added to
  timestamps. This explains why some identifier columns show one dominant value shared
  across many organizations (Section 8) rather than true nulls.
- **Target labels:** `IncidentGrade` is the customer-provided triage label, `TruePositive`,
  `BenignPositive`, `FalsePositive` , the primary benchmark target (Section 6).
- **Official train/test split convention:** splits are assigned at the **organization**
  level, so incidents from the same org never appear across multiple splits. Relevant to
  the multi-org `IncidentId` check in Section 7 and to the modeling notebook's split logic.
- **Scale:** 13M+ data points, 33 entity types, 1.6M alerts, ~1M labeled incidents, 6,100+
  organizations, 441 MITRE ATT&CK techniques.


## 1. Setup

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association

import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [2]:
df = pd.read_csv('/kaggle/input/datasets/organizations/Microsoft/microsoft-security-incident-prediction/GUIDE_Train.csv')
# df_test = pd.read_csv('/kaggle/input/datasets/organizations/Microsoft/microsoft-security-incident-prediction/GUIDE_Test.csvv')

In [3]:
df.head()

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04T06:05:15.000Z,7,6,InitialAccess,NaN,TruePositive,NaN,NaN,Ip,Related,98799,138268,27,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,31,6,3
1,455266534868,88,326,210035,2024-06-14T03:01:25.000Z,58,43,Exfiltration,NaN,FalsePositive,NaN,NaN,User,Impacted,98799,138268,360606,160396,22406,23032,22795,24887,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
2,1056561957389,809,58352,712507,2024-06-13T04:52:55.000Z,423,298,InitialAccess,T1189,FalsePositive,NaN,NaN,Url,Related,98799,138268,360606,68652,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
3,1279900258736,92,32992,774301,2024-06-10T16:39:36.000Z,2,2,CommandAndControl,NaN,BenignPositive,NaN,NaN,Url,Related,98799,138268,360606,13,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
4,214748368522,148,4359,188041,2024-06-15T01:08:07.000Z,9,74,Execution,NaN,TruePositive,NaN,NaN,User,Impacted,98799,138268,360606,160396,449,592,440,479,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


In [4]:
df.shape

(9516837, 45)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9516837 entries, 0 to 9516836
Data columns (total 45 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Id                  int64  
 1   OrgId               int64  
 2   IncidentId          int64  
 3   AlertId             int64  
 4   Timestamp           object 
 5   DetectorId          int64  
 6   AlertTitle          int64  
 7   Category            object 
 8   MitreTechniques     object 
 9   IncidentGrade       object 
 10  ActionGrouped       object 
 11  ActionGranular      object 
 12  EntityType          object 
 13  EvidenceRole        object 
 14  DeviceId            int64  
 15  Sha256              int64  
 16  IpAddress           int64  
 17  Url                 int64  
 18  AccountSid          int64  
 19  AccountUpn          int64  
 20  AccountObjectId     int64  
 21  AccountName         int64  
 22  DeviceName          int64  
 23  NetworkMessageId    int64  
 24  EmailClusterId      floa

## Dataset Column Groups

For better understanding, the data is grouped into different categories.

### 1. Identifiers (Keys)
- `Id` - int64
- `OrgId` - int64
- `IncidentId` - int64
- `AlertId` - int64

### 2. Timing
- `Timestamp` - object *(likely needs `pd.to_datetime` conversion)*

### 3. Detection / Classification
- `DetectorId` - int64
- `AlertTitle` - int64 *(encoded as int, likely a hashed/label-encoded category, not text)*
- `Category` - object
- `MitreTechniques` - object
- `IncidentGrade` - object *(likely the target/label column)*
- `ActionGrouped` - object
- `ActionGranular` - object
- `SuspicionLevel` - object
- `LastVerdict` - object

### 4. Entity Context (what the alert is about)
- `EntityType` - object
- `EvidenceRole` - object

### 5. Device / Host Identifiers
- `DeviceId` - int64
- `DeviceName` - int64
- `OSFamily` - int64 *(encoded, not the string family name)*
- `OSVersion` - int64 *(encoded)*

### 6. File / Threat Artifacts
- `Sha256` - int64
- `FileName` - int64
- `FolderPath` - int64
- `ThreatFamily` - object

### 7. Network / Web Artifacts
- `IpAddress` - int64
- `Url` - int64
- `NetworkMessageId` - int64
- `EmailClusterId` - float64
- `AntispamDirection` - object

### 8. Identity / Account Artifacts
- `AccountSid` - int64
- `AccountUpn` - int64
- `AccountObjectId` - int64
- `AccountName` - int64
- `Roles` - object

### 9. Registry Artifacts (Windows-specific)
- `RegistryKey` - int64
- `RegistryValueName` - int64
- `RegistryValueData` - int64

### 10. Application / OAuth Artifacts
- `ApplicationId` - int64
- `ApplicationName` - int64
- `OAuthApplicationId` - int64

### 11. Resource Metadata
- `ResourceIdName` - int64
- `ResourceType` - object

### 12. Geography
- `CountryCode` - int64
- `State` - int64
- `City` - int64

In [6]:
for col in df.columns:
    print(col, "|", df[col].dtype, "| nunique:", df[col].nunique(), "| nulls:", df[col].isna().sum())

Id | int64 | nunique: 730778 | nulls: 0
OrgId | int64 | nunique: 5769 | nulls: 0
IncidentId | int64 | nunique: 466151 | nulls: 0
AlertId | int64 | nunique: 1265644 | nulls: 0
Timestamp | object | nunique: 760944 | nulls: 0
DetectorId | int64 | nunique: 8428 | nulls: 0
AlertTitle | int64 | nunique: 86149 | nulls: 0
Category | object | nunique: 20 | nulls: 0
MitreTechniques | object | nunique: 1193 | nulls: 5468386
IncidentGrade | object | nunique: 3 | nulls: 51340
ActionGrouped | object | nunique: 3 | nulls: 9460773
ActionGranular | object | nunique: 16 | nulls: 9460773
EntityType | object | nunique: 33 | nulls: 0
EvidenceRole | object | nunique: 2 | nulls: 0
DeviceId | int64 | nunique: 75826 | nulls: 0
Sha256 | int64 | nunique: 106416 | nulls: 0
IpAddress | int64 | nunique: 285957 | nulls: 0
Url | int64 | nunique: 123252 | nulls: 0
AccountSid | int64 | nunique: 358401 | nulls: 0
AccountUpn | int64 | nunique: 530183 | nulls: 0
AccountObjectId | int64 | nunique: 343516 | nulls: 0
Account

## 4. Verification of Data Types

In [7]:
for col in ['OSFamily', 'CountryCode', 'AlertTitle', 'Sha256', 'IpAddress', 'DeviceId', 'AccountObjectId']:
    print(col, df[col].min(), df[col].max(), df[col].unique()[:5])

OSFamily 0 5 [5 0 2 1 3]
CountryCode 0 242 [ 31 242   8   7  39]
AlertTitle 0 113174 [  6  43 298   2  74]
Sha256 0 138268 [138268      0      4  10713  12379]
IpAddress 0 360606 [    27 360606  30410    279   8146]
DeviceId 0 98799 [98799  2504 17769  5291    84]
AccountObjectId 0 425863 [425863  22795    440   4086 173595]


In [8]:
for col in ['Sha256', 'IpAddress', 'DeviceId', 'AccountObjectId', 'AlertTitle']:
    print(col, '-> % zero:', (df[col] == 0).mean())

Sha256 -> % zero: 0.005304598576186605
IpAddress -> % zero: 0.001312305758730553
DeviceId -> % zero: 0.0004957529481696493
AccountObjectId -> % zero: 0.0015203580769535088
AlertTitle -> % zero: 0.13998684647010345


**Observation:** `AlertTitle` has a notably high share of zeros (~14%) compared to the other
identifier columns checked above (all well under 1%). Since it's a hashed/encoded column,
it's unclear at this point whether `0` is a real, meaningful category or should be treated
as missing (`NaN`). Checked further below.

In [9]:
df.groupby('EntityType')['AlertTitle'].apply(lambda x: (x == 0).mean()).sort_values(ascending=False)

EntityType
CloudLogonSession        0.916650
CloudLogonRequest        0.593390
User                     0.196440
Ip                       0.173772
ActiveDirectoryDomain    0.000000
BlobContainer            0.000000
Blob                     0.000000
CloudApplication         0.000000
AmazonResource           0.000000
Container                0.000000
ContainerImage           0.000000
File                     0.000000
ContainerRegistry        0.000000
GenericEntity            0.000000
GoogleCloudResource      0.000000
IoTDevice                0.000000
AzureResource            0.000000
KubernetesCluster        0.000000
KubernetesNamespace      0.000000
Machine                  0.000000
KubernetesPod            0.000000
MailMessage              0.000000
Mailbox                  0.000000
MailboxConfiguration     0.000000
MailCluster              0.000000
Malware                  0.000000
Nic                      0.000000
Process                  0.000000
OAuthApplication         0.000000
Reg

**Result:** zero-rate is concentrated in `CloudLogonSession` (91.7%), `CloudLogonRequest`
(59.3%), `User` (19.6%), and `Ip` (17.4%) — every other entity type is 0%.

In [10]:
df[df['AlertTitle'] == 0]['Category'].value_counts()

Category
InitialAccess    1332232
Name: count, dtype: int64

**Conclusion : `AlertTitle == 0` is a real, meaningful value, not missing data.** Two
findings together confirm this:

- By `EntityType`, the zero-rate is concentrated almost entirely in cloud logon-related
  entities - `CloudLogonSession` (91.7%), `CloudLogonRequest` (59.3%), `User` (19.6%),
  `Ip` (17.4%) - while every other entity type has a **0% zero-rate**. This isn't random
  or spread-out missingness; it's tied to a specific subset of entity types.
- Filtering to only `AlertTitle == 0` rows and checking `Category` shows **100% of them
  are `InitialAccess`** (1,332,232 out of 1,332,232 rows) , not just a majority, all of them.

Together this means `0` is one specific, very common, real alert title - most likely
something like a routine cloud logon alert , that is always categorized as `InitialAccess`.
It is not an error, placeholder, or ambiguous encoding.

**Decision: keep `AlertTitle` as-is, `0` included, no NaN conversion needed.** This column
is *not* flagged for the modeling include/exclude test , the EDA here is conclusive enough
to resolve it now rather than deferring the decision.

The broader null/zero pattern across the *other* artifact columns (unrelated to
`AlertTitle`) remains expected `EntityType` determines which artifact fields are
relevant for a given row, so many artifact columns are legitimately empty depending on
entity type. That's a structural feature of the data, not a data-quality problem.

## 5. Dtype Fix — `Timestamp`

In [11]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

## 6. Target Variable — `IncidentGrade`

In [12]:
print(df['IncidentGrade'].info())
print(df['IncidentGrade'].value_counts())
print(df['IncidentGrade'].value_counts(normalize=True))

<class 'pandas.core.series.Series'>
RangeIndex: 9516837 entries, 0 to 9516836
Series name: IncidentGrade
Non-Null Count    Dtype 
--------------    ----- 
9465497 non-null  object
dtypes: object(1)
memory usage: 72.6+ MB
None
IncidentGrade
BenignPositive    4110817
TruePositive      3322713
FalsePositive     2031967
Name: count, dtype: int64
IncidentGrade
BenignPositive    0.434295
TruePositive      0.351034
FalsePositive     0.214671
Name: proportion, dtype: float64


**Observation:** approximately 500,000 rows have a null `IncidentGrade`. Among labeled rows,
the class split is **43.43% BenignPositive / 35.10% TruePositive / 21.47% FalsePositive**.
This is a moderate class imbalance worth accounting for in modeling (e.g., stratified
splits, class weighting, or resampling).

In [13]:
df['IncidentGrade'].groupby([df['IncidentId']]).value_counts(normalize = True).head(20)

IncidentId  IncidentGrade 
0           TruePositive      1.000000
2           BenignPositive    0.999367
            TruePositive      0.000633
3           TruePositive      1.000000
7           BenignPositive    0.573294
            FalsePositive     0.424502
            TruePositive      0.002204
8           BenignPositive    0.500000
            TruePositive      0.500000
9           FalsePositive     0.999914
            BenignPositive    0.000086
10          BenignPositive    0.789474
            FalsePositive     0.210526
11          FalsePositive     0.986842
            BenignPositive    0.006579
            TruePositive      0.006579
13          BenignPositive    1.000000
14          TruePositive      1.000000
17          BenignPositive    1.000000
19          BenignPositive    1.000000
Name: proportion, dtype: float64

In [14]:
df['IncidentId'].nunique()

466151

In [15]:
df[df['IncidentId'] == 7]

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
491,987842482063,14,7,622,2024-05-23 19:06:24+00:00,41,84,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,153,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
719,790273987737,104,7,752,2024-06-03 11:49:19+00:00,11,9,InitialAccess,T1566,FalsePositive,NaN,NaN,User,Impacted,98799,138268,360606,160396,128768,163498,132604,153920,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
1234,790273987737,104,7,3191,2024-06-03 13:28:11+00:00,11,9,InitialAccess,T1566,FalsePositive,NaN,NaN,MailMessage,Related,98799,138268,360606,160396,441377,71719,425863,453297,153085,294139,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
3420,987842482063,14,7,95,2024-05-24 00:08:14+00:00,41,183,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,782,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
3902,987842482063,14,7,164,2024-05-24 05:10:22+00:00,41,58,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,3748,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9508463,987842482063,14,7,223,2024-05-24 06:06:20+00:00,41,84,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,297,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
9510306,790273987737,104,7,285,2024-06-03 12:02:21+00:00,11,9,InitialAccess,T1566,FalsePositive,NaN,NaN,Mailbox,Impacted,98799,138268,360606,160396,45484,46092,41787,46369,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
9513228,790273987737,104,7,2308,2024-06-03 12:43:55+00:00,11,9,InitialAccess,T1566,FalsePositive,NaN,NaN,MailMessage,Related,98799,138268,360606,160396,441377,81185,425863,453297,153085,202321,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
9515728,790273987737,104,7,3508,2024-06-03 09:37:47+00:00,11,9,InitialAccess,T1566,FalsePositive,NaN,NaN,Mailbox,Impacted,98799,138268,360606,160396,67439,76529,78881,75434,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


## 7. Incident Structure — Multi-Org Incidents

Checking whether a single `IncidentId` can span multiple organizations (`OrgId`), which
would affect how rows should be grouped/deduplicated and how leakage is avoided in
train/test splits.

In [16]:
df.groupby('IncidentId')['OrgId'].nunique().value_counts()

OrgId
1     334863
2      67088
3      29575
4      16297
5       9152
6       4898
7       2530
8       1086
9        442
10       148
11        49
12        16
13         7
Name: count, dtype: int64

**Observation:** a small number of `IncidentId`s appear across more than one `OrgId`. Worth
keeping in mind for any train/test split strategy (grouping by `IncidentId` alone won't
guarantee no organization-level leakage across splits) , revisit when building the
train/test split in the modeling notebook.

## 8. Which Columns Have a Hidden "Missing Value" Sentinel?

Many int-encoded artifact columns (`IpAddress`, `RegistryKey`, `FolderPath`, `FileName`,
`AccountUpn`, etc.) are only relevant for specific `EntityType`s , an `IpAddress` field
means something for an `Ip` entity, but not for a `File` entity. The question this section
answers: when the field isn't relevant, what value does it actually contain in the raw
data? If it isn't a true `NaN`, is it landing on one specific dominant value instead and
if so, is that dominant value a real, meaningful measurement, or a placeholder standing in
for "not applicable here"? Each candidate column is checked and the conclusion is backed
by direct evidence, not assumption, before anything gets converted.

In [17]:
df[df['Id'] == 987842482063]

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
491,987842482063,14,7,622,2024-05-23 19:06:24+00:00,41,84,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,153,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
3420,987842482063,14,7,95,2024-05-24 00:08:14+00:00,41,183,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,782,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
3902,987842482063,14,7,164,2024-05-24 05:10:22+00:00,41,58,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,3748,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
6083,987842482063,14,7,95,2024-05-24 00:08:14+00:00,41,183,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,4036,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
6135,987842482063,14,7,795,2024-05-23 08:27:20+00:00,41,985,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,1298,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9496283,987842482063,14,7,257,2024-05-24 06:20:50+00:00,41,387,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,7079,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
9497792,987842482063,14,7,795,2024-05-23 08:27:20+00:00,41,985,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,5321,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
9498730,987842482063,14,7,81,2024-05-23 23:37:03+00:00,41,700,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,10448,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
9502876,987842482063,14,7,102,2024-05-24 09:45:49+00:00,41,724,Impact,NaN,BenignPositive,NaN,NaN,Ip,Related,98799,138268,110,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,NaN,289573,117668,3586,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


### 8.1 Step one: find each column's dominant ("mode") value

For each candidate column, find the single most frequent value and what share of all rows
it makes up. A very high share is the first sign but only a sign, not proof that a
column might be using one value as a stand-in for "not applicable" rather than a real
measurement.

In [18]:
candidate_cols = [
    # Included so AlertTitle gets the same systematic check as everything else
    # (already resolved in Section 4, kept here for a single consistent view):
    'AlertTitle',
    # Already checked, kept here for a single consistent run:
    'RegistryKey', 'City', 'IpAddress', 'FolderPath', 'ResourceIdName',
    'OSFamily', 'OSVersion', 'CountryCode', 'State',
    'RegistryValueName', 'RegistryValueData', 'AccountUpn', 'FileName',
    # Newly added — same kind of entity-specific artifact column, not yet checked:
    'DeviceId', 'DeviceName', 'Sha256', 'Url', 'NetworkMessageId', 'EmailClusterId',
    'AccountSid', 'AccountObjectId', 'AccountName',
    'ApplicationId', 'ApplicationName', 'OAuthApplicationId',
]

mode_values = {}
for col in candidate_cols:
    top_val = df[col].mode()[0]
    top_share = (df[col] == top_val).mean()
    mode_values[col] = top_val
    print(col, '-> mode:', top_val, '| % of rows:', round(top_share, 4))

AlertTitle -> mode: 0 | % of rows: 0.14
RegistryKey -> mode: 1631 | % of rows: 0.9982
City -> mode: 10630 | % of rows: 0.9334
IpAddress -> mode: 360606 | % of rows: 0.7712
FolderPath -> mode: 117668 | % of rows: 0.908
ResourceIdName -> mode: 3586 | % of rows: 0.9991
OSFamily -> mode: 5 | % of rows: 0.9796
OSVersion -> mode: 66 | % of rows: 0.9796
CountryCode -> mode: 242 | % of rows: 0.9212
State -> mode: 1445 | % of rows: 0.9335
RegistryValueName -> mode: 635 | % of rows: 0.9995
RegistryValueData -> mode: 860 | % of rows: 0.9994
AccountUpn -> mode: 673934 | % of rows: 0.636
FileName -> mode: 289573 | % of rows: 0.8919
DeviceId -> mode: 98799 | % of rows: 0.9619
DeviceName -> mode: 153085 | % of rows: 0.9267
Sha256 -> mode: 138268 | % of rows: 0.923
Url -> mode: 160396 | % of rows: 0.9283
NetworkMessageId -> mode: 529644 | % of rows: 0.8782
EmailClusterId -> mode: 4218786605.0 | % of rows: 0.0002
AccountSid -> mode: 441377 | % of rows: 0.7631
AccountObjectId -> mode: 425863 | % of rows

### 8.2 Step two: get actual proof via the `EntityType` breakdown

A high mode share alone isn't proof of a placeholder a genuinely common, low-cardinality
value (e.g., a country or OS family most machines really do share) can look identical on
this metric alone. The real test is to break the mode-match rate down by `EntityType`:

- A **true "not applicable" sentinel** should be **rare** specifically for the one entity
  type where the column is actually meaningful (e.g., real IP addresses vary a lot among
  `Ip` rows), and **universal** everywhere else (nothing real to put there, so it defaults
  to the same placeholder).
- A **genuine, common real value** would not show this inversion it would just look
  fairly high everywhere, including in its "relevant" entity type, because it's simply an
  accurate, frequently-occurring measurement.

Rather than print all ~30 entity types per column, only the entity types that **break**
from near-100% are printed below those are the only rows that actually matter for
spotting the inversion pattern. Anything not listed for a column can be assumed to sit at
(or very near) 100% match.

In [19]:
EXCEPTION_THRESHOLD = 0.99  # entity types below this rate are shown; everything else is ~100%

for col in candidate_cols:
    mode_val = mode_values[col]
    rates = df.groupby('EntityType')[col].apply(lambda x: (x == mode_val).mean())
    exceptions = rates[rates < EXCEPTION_THRESHOLD].sort_values(ascending=False)
    print(f"--- {col} == {mode_val} ---")
    if exceptions.empty:
        print("  (no exceptions — every EntityType is ~100% this value)")
    else:
        print(exceptions)
    print()

--- AlertTitle == 0 ---
EntityType
CloudLogonSession        0.916650
CloudLogonRequest        0.593390
User                     0.196440
Ip                       0.173772
ActiveDirectoryDomain    0.000000
BlobContainer            0.000000
Blob                     0.000000
CloudApplication         0.000000
AmazonResource           0.000000
Container                0.000000
ContainerImage           0.000000
File                     0.000000
ContainerRegistry        0.000000
GenericEntity            0.000000
GoogleCloudResource      0.000000
IoTDevice                0.000000
AzureResource            0.000000
KubernetesCluster        0.000000
KubernetesNamespace      0.000000
Machine                  0.000000
KubernetesPod            0.000000
MailMessage              0.000000
Mailbox                  0.000000
MailboxConfiguration     0.000000
MailCluster              0.000000
Malware                  0.000000
Nic                      0.000000
Process                  0.000000
OAuthApplicat

### 8.3 Result: four groups

**Already resolved separately** - `AlertTitle`. Checked in Section 4: its dominant value
(`0`) shows the *opposite* of the sentinel pattern it's concentrated (91.6% at
`CloudLogonSession`, 59.4% at `CloudLogonRequest`) rather than universal everywhere, and
maps 100% to `Category == InitialAccess`. Confirmed as a real, meaningful value kept
as-is, not converted. Included in the 8.1/8.2 scan above only for a single consistent view.

**Confirmed sentinels**  mode value is rare specifically where the column should be real,
matching the clean inversion signature described above:

| Column | Relevant entity type | Rate there |
|---|---|---|
| `IpAddress` (360606) | `Ip` | 0.16% (`Nic` 0%) |
| `ResourceIdName` (3586) | `AzureResource` | 0% |
| `FileName` (289573) | `Process` 1.4%, `File` 0.03% |
| `FolderPath` (117668) | `File` 22.3%, `Process` 1.5% |
| `RegistryKey` (1631) | `RegistryValue` 8.5%, `RegistryKey` 0.14% |
| `AccountUpn` (673934) | `User` 6.0%, `Mailbox` 1.5%, `MailMessage` 0.16% (`MailboxConfiguration` 70.8% is a softer spot, not fully clean, but the majority of relevant types confirm) |
| `Url` (160396) | `Url` | 0.02% |
| `NetworkMessageId` (529644) | `MailMessage` | 1.2% |
| `DeviceName` (153085) | `Machine` | 0.36% |
| `ApplicationId` (2251) | `CloudApplication` | 1.8% |
| `ApplicationName` (3421) | `CloudApplication`/`OAuthApplication` | 0.09% / 0% |
| `OAuthApplicationId` (881) | `OAuthApplication` | 0% |
| `AccountSid` (441377) | `User`/`Mailbox` | 6.4% / 7.7% |
| `AccountObjectId` (425863) | `User`/`Mailbox` | 8.0% / 6.4% |
| `AccountName` (453297) | `Mailbox`/`User` | 7.7% / 0.6% |

These all show the actual proof needed: the mode value nearly disappears exactly where a
real value should exist, and dominates everywhere else. That's the direct evidence for
treating them as "not applicable" sentinels, not just a hunch from a high mode share.

**Not confirmed : left unmodified:** `City`, `State`, `CountryCode`, `OSFamily`,
`OSVersion`, `RegistryValueName`, `RegistryValueData`, `DeviceId`, `Sha256`. For these, the
"relevant" entity type still shows the mode value at a substantial rate (50–94%) rather
than the near-zero rate the confirmed group shows - a much weaker inversion, not treated
as proven. `DeviceId` (`Machine` 48.2%) and `Sha256` (`File` 42.9%) were checked directly
and came back with rates far too high to count as a clean sentinel signature. No further
investigation planned for these - carried into modeling unchanged.

**Already clean, no action needed** - `EmailClusterId`. Its highest rate anywhere is 0.89%
(`MailCluster`) - nowhere close to dominant, so there's no hidden sentinel to convert. This
column already uses explicit `NaN` for missing values (per Section 3), which is exactly
the correct, non-disguised form the other confirmed columns are being converted *to*.

### 8.4 Applying the confirmed conversion

Only the six columns with actual proof from 8.3 are converted. Two versions of the
dataframe are produced so the conversion itself can be tested during modeling:

- `df_clean_sentinel_nan` - confirmed sentinel values replaced with `NaN`
- `df_clean_no_conversion` - left in place, untouched

Everything else (duplicate removal, `Timestamp` conversion) is applied identically to
both, so this conversion is the *only* difference between the two output files.

In [20]:
confirmed_sentinels = {
    'IpAddress': 360606,
    'ResourceIdName': 3586,
    'FileName': 289573,
    'FolderPath': 117668,
    'RegistryKey': 1631,
    'AccountUpn': 673934,
    'Url': 160396,
    'NetworkMessageId': 529644,
    'DeviceName': 153085,
    'ApplicationId': 2251,
    'ApplicationName': 3421,
    'OAuthApplicationId': 881,
    'AccountSid': 441377,
    'AccountObjectId': 425863,
    'AccountName': 453297,
}

df_clean_sentinel_nan = df.copy()
for col, val in confirmed_sentinels.items():
    df_clean_sentinel_nan[col] = df_clean_sentinel_nan[col].replace(val, np.nan)

df_clean_no_conversion = df.copy()

print("Sentinel-to-NaN version — new null counts for converted columns:")
print(df_clean_sentinel_nan[list(confirmed_sentinels.keys())].isna().sum())

Sentinel-to-NaN version — new null counts for converted columns:
IpAddress             7339164
ResourceIdName        9508671
FileName              8487890
FolderPath            8641204
RegistryKey           9499313
AccountUpn            6052546
Url                   8834131
NetworkMessageId      8357998
DeviceName            8819646
ApplicationId         9303934
ApplicationName       9297632
OAuthApplicationId    9514242
AccountSid            7262455
AccountObjectId       7286295
AccountName           7150260
dtype: int64


## Checking for duplicates

In [21]:
df.duplicated().sum()

np.int64(22559)

In [22]:
df[df.duplicated(keep=False)]['IncidentGrade'].value_counts(normalize=True)

IncidentGrade
TruePositive      0.859077
FalsePositive     0.136989
BenignPositive    0.003934
Name: proportion, dtype: float64

**Decision:** for this study, duplicate rows will be removed they amount to less than
0.5% of the total data. In a real-world setting, this would be worth a second look, since
there are two plausible explanations:

1. These are genuinely repeated alerts for the same simultaneous attack (worth confirming
   with an IT/SOC analyst, especially since duplicated rows skew ~85% `TruePositive`), or
2. These are data-collection errors and should simply be removed.

Given the small share of affected rows, we proceed with removal for this analysis. Applied
identically to both cleaned versions.

In [23]:
df_clean_sentinel_nan = df_clean_sentinel_nan.drop_duplicates()
df_clean_no_conversion = df_clean_no_conversion.drop_duplicates()

print("Sentinel-to-NaN version duplicates remaining:", df_clean_sentinel_nan.duplicated().sum())
print("No-conversion version duplicates remaining:", df_clean_no_conversion.duplicated().sum())

Sentinel-to-NaN version duplicates remaining: 0
No-conversion version duplicates remaining: 0


In [24]:
df_clean_sentinel_nan.to_csv('/kaggle/working/GUIDE_Train_clean_sentinel_nan.csv', index=False)
df_clean_no_conversion.to_csv('/kaggle/working/GUIDE_Train_clean_no_conversion.csv', index=False)